# 01 数据理解：Scania APS 预测性维护项目

本 notebook 用于 Day 1 的项目初始化和数据理解准备。当前阶段只确认业务背景、数据路径、基础读取方式和标签含义，不进行复杂清洗、特征工程、模型训练或阈值调优。

业务成本设定：误报 FP 成本为 10，漏报 FN 成本为 500。后续评估不会以 accuracy 作为核心目标，而会重点关注 precision、recall、F1、F2、PR-AUC 和 total cost。

## 1. 导入依赖

这里只导入 Day 1 数据读取和基础查看需要的依赖。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 2. 设置数据路径

原始数据保存在 `data/raw/`。Day 1 只读取原始 train/test 文件，不修改原始数据，也不合并后重新切分。

In [2]:
# 支持从项目根目录或 notebooks/ 目录启动 notebook。
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_training_set.csv"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "aps_failure_test_set.csv"

TRAIN_PATH, TEST_PATH

(WindowsPath('c:/Scania APS/data/raw/aps_failure_training_set.csv'),
 WindowsPath('c:/Scania APS/data/raw/aps_failure_test_set.csv'))

## 3. 读取 train/test 数据

原始 CSV 中的字符串 `na` 表示缺失值，读取时需要通过 `na_values="na"` 转换为 pandas 缺失值。

In [3]:
train_df = pd.read_csv(TRAIN_PATH, na_values="na")
test_df = pd.read_csv(TEST_PATH, na_values="na")

train_df.head()

,class,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,...,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,76698,NaN,2.130706e+09,280.0,0.0,0.0,0.0,0.0,0.0,...,1240520.0,493384.0,721044.0,469792.0,339156.0,157956.0,73224.0,0.0,0.0,0.0
1,neg,33058,NaN,0.000000e+00,NaN,0.0,0.0,0.0,0.0,0.0,...,421400.0,178064.0,293306.0,245416.0,133654.0,81140.0,97576.0,1500.0,0.0,0.0
2,neg,41040,NaN,2.280000e+02,100.0,0.0,0.0,0.0,0.0,0.0,...,277378.0,159812.0,423992.0,409564.0,320746.0,158022.0,95128.0,514.0,0.0,0.0
3,neg,12,0.0,7.000000e+01,66.0,0.0,10.0,0.0,0.0,0.0,...,240.0,46.0,58.0,44.0,10.0,0.0,0.0,0.0,4.0,32.0
4,neg,60874,NaN,1.368000e+03,458.0,0.0,0.0,0.0,0.0,0.0,...,622012.0,229790.0,405298.0,347188.0,286954.0,311560.0,433954.0,1218.0,0.0,0.0


## 4. 查看数据规模

这里只查看行数和列数，不对数据作清洗决策。

In [4]:
shape_summary = pd.DataFrame(
    {
        "数据集": ["train", "test"],
        "行数": [train_df.shape[0], test_df.shape[0]],
        "列数": [train_df.shape[1], test_df.shape[1]],
    }
)

shape_summary

,数据集,行数,列数
0,train,60000,171
1,test,16000,171


## 5. 查看字段信息

字段名经过匿名化处理，Day 1 先确认字段数量、字段类型和标签列位置。

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 171 entries, class to eg_000
dtypes: float64(169), int64(1), object(1)
memory usage: 78.3+ MB


In [6]:
train_df.columns[:10].tolist(), train_df.columns[-10:].tolist()

(['class',
  'aa_000',
  'ab_000',
  'ac_000',
  'ad_000',
  'ae_000',
  'af_000',
  'ag_000',
  'ag_001',
  'ag_002'],
 ['ee_002',
  'ee_003',
  'ee_004',
  'ee_005',
  'ee_006',
  'ee_007',
  'ee_008',
  'ee_009',
  'ef_000',
  'eg_000'])

## 6. 查看 class 标签分布

`pos` 表示 APS 系统相关故障，`neg` 表示非 APS 系统相关故障。这里先查看分布，为后续类别不平衡分析做准备。

In [7]:
label_distribution = (
    train_df["class"]
    .value_counts(dropna=False)
    .rename_axis("class")
    .reset_index(name="样本数")
)

label_distribution["占比"] = label_distribution["样本数"] / len(train_df)
label_distribution

,class,样本数,占比
0,neg,59000,0.983333
1,pos,1000,0.016667


## 7. 将 class 映射为 target

后续建模会使用 `target`：`pos -> 1`，`neg -> 0`。Day 1 只创建映射列用于理解标签，不做模型训练。

In [8]:
target_mapping = {"neg": 0, "pos": 1}

train_df = train_df.assign(target=train_df["class"].map(target_mapping))
test_df = test_df.assign(target=test_df["class"].map(target_mapping))

train_df[["class", "target"]].head()

,class,target
0,neg,0
1,neg,0
2,neg,0
3,neg,0
4,neg,0


In [9]:
train_df["target"].value_counts(dropna=False).sort_index()

target
0    59000
1     1000
Name: count, dtype: int64

## Day 1 小结

- 已明确项目业务目标：预测 APS 系统相关故障，并服务维修优先级决策。
- 已明确标签映射：`pos -> 1`，`neg -> 0`。
- 已明确成本设定：FP = 10，FN = 500。
- 已确认原始数据读取方式：`pd.read_csv(..., na_values="na")`。
- 当前阶段不进行模型训练、复杂清洗、特征工程或阈值选择。